## GroupBy & Aggregation

### Why does this matter?

GroupBy is the single most important Pandas operation for a Data Analyst. Every business question that starts with **"per" or "by"** needs GroupBy — revenue per region, average salary by department, total orders per customer. If you can't do GroupBy confidently, you can't do data analysis.

### What is GroupBy?

**What is it?** GroupBy splits your DataFrame into groups based on one or more columns, applies a function to each group separately, and combines the results back into a single output. This is called the **Split → Apply → Combine pattern.**

### Why does it exist?

Because you almost never want statistics on the whole dataset. You want statistics per group. "What is the average salary?" is a weak question. "What is the average salary per department?" is what business decisions are made on.

### How does it work internally?

When you call **df.groupby('Dept')**, Pandas doesn't compute anything yet. It creates a **GroupBy** object — a blueprint that knows how to split the data. Only when you chain an aggregation like **.mean() or .sum()** does the actual computation happen. This is called lazy evaluation — work is deferred until needed.

In [1]:
import pandas as pd

data = {
    'Name':   ['Sumed','Priya','Rahul','Neha','Arjun','Divya'],
    'Dept':   ['Data', 'HR',  'Data', 'Finance','Data', 'HR'  ],
    'Salary': [60000,45000,75000,55000,  80000,50000],
    'Exp':    [3,     2,    5,     4,       6,     3    ]
}
df = pd.DataFrame(data)
print(df)

    Name     Dept  Salary  Exp
0  Sumed     Data   60000    3
1  Priya       HR   45000    2
2  Rahul     Data   75000    5
3   Neha  Finance   55000    4
4  Arjun     Data   80000    6
5  Divya       HR   50000    3


## The SPLIT>APPLY>COMBINE PATTERN

In [3]:
# STEP 1: groupby() — SPLIT the df into groups
# Nothing is computed yet. Just creates a GroupBy object.
grouped = df.groupby('Dept')
print(type(grouped))
# → DataFrameGroupBy object (not a DataFrame yet!)



<class 'pandas.core.groupby.generic.DataFrameGroupBy'>


In [5]:
# STEP 2: APPLY an aggregation function to each group
grouped['Salary'].mean()   # mean salary per dept


Dept
Data       71666.666667
Finance    55000.000000
HR         47500.000000
Name: Salary, dtype: float64

In [6]:
grouped['Salary'].sum()    # total salary per dept


Dept
Data       215000
Finance     55000
HR          95000
Name: Salary, dtype: int64

In [7]:
grouped['Salary'].max()    # highest salary per dept


Dept
Data       80000
Finance    55000
HR         50000
Name: Salary, dtype: int64

In [8]:
grouped['Salary'].min()    # lowest salary per dept


Dept
Data       60000
Finance    55000
HR         45000
Name: Salary, dtype: int64

In [9]:
grouped['Salary'].count()  # headcount per dept

# STEP 3: Result is COMBINED into a new Series/DataFrame
# The group key (Dept) becomes the new index

Dept
Data       3
Finance    1
HR         2
Name: Salary, dtype: int64

*Dept is now the index of the result — because it was the grouping key. This is important — the result is a Series, not a DataFrame.*

## Groupby on entire DataFrame

In [11]:
# When you don't select a specific column → applies to ALL numeric cols
# df.groupby('Dept').mean()       # mean of Salary AND Exp per Dept
df.groupby('Dept').sum()        # sum of all numeric columns
df.groupby('Dept').count()      # count non-null values per column

# numeric_only=True — avoids warnings in newer Pandas versions
# explicitly tells Pandas to skip non-numeric columns
df.groupby('Dept').mean(numeric_only=True)

,Salary,Exp
Dept,,
Data,71666.666667,4.666667
Finance,55000.000000,4.000000
HR,47500.000000,2.500000


## AGG() : Multiple AGGREGATION at once

In [13]:
# agg() → apply MULTIPLE functions at once
# Returns a DataFrame (not just a Series)
# This is what you use in REAL analysis — not just one stat

# Same function to multiple columns
df.groupby('Dept')['Salary'].agg(['mean', 'min', 'max', 'count'])

# Different functions to different columns
df.groupby('Dept').agg({
    'Salary': ['mean', 'max'],   # two stats for Salary
    'Exp':    ['min', 'sum']    # two stats for Exp
})



Salary        Exp    
                 mean    max min sum
Dept                                
Data     71666.666667  80000   3  14
Finance  55000.000000  55000   4   4
HR       47500.000000  50000   2   5

In [14]:
# Named aggregations — cleaner column names in output
df.groupby('Dept').agg(
    avg_salary = ('Salary', 'mean'),
    max_salary = ('Salary', 'max'),
    headcount  = ('Name',   'count')
)

,avg_salary,max_salary,headcount
Dept,,,
Data,71666.666667,80000,3
Finance,55000.000000,55000,1
HR,47500.000000,50000,2


*Named aggregation syntax — new_col_name = ('source_col', 'function') — is the most professional way. Output column names are clean and meaningful. Use this in interviews.*

## Groupby on Multiple column

In [16]:
# Group by MORE than one column → creates a MultiIndex result
# Real use: "average salary by Dept AND experience level"

# Add a seniority column first
df['Level'] = df['Exp'].apply(lambda x: 'Senior' if x > 4 else 'Junior')



In [17]:
# Group by two columns
df.groupby(['Dept', 'Level'])['Salary'].mean()


Dept     Level 
Data     Junior    60000.0
         Senior    77500.0
Finance  Junior    55000.0
HR       Junior    47500.0
Name: Salary, dtype: float64

In [18]:

# as_index=False → keeps group keys as regular columns
# instead of making them the index (more DataFrame-friendly)
df.groupby(['Dept', 'Level'], as_index=False)['Salary'].mean()

,Dept,Level,Salary
0,Data,Junior,60000.0
1,Data,Senior,77500.0
2,Finance,Junior,55000.0
3,HR,Junior,47500.0


 *as_index=False is the same as doing .reset_index() at the end. Use it when you want to keep working with the result as a normal DataFrame — which is almost always.*

## TRANSFORM(): keep original DF shape

In [19]:
# agg() shrinks the df → one row per group
# transform() keeps the SAME shape as original df
# It broadcasts the group result back to every row in that group

# Add a column: dept average salary (same row count as original)
df['Dept_Avg_Salary'] = df.groupby('Dept')['Salary'].transform('mean')

# Now you can compare each person's salary to their dept average
df['Above_Avg'] = df['Salary'] > df['Dept_Avg_Salary']

print(df[['Name', 'Dept', 'Salary', 'Dept_Avg_Salary', 'Above_Avg']])

    Name     Dept  Salary  Dept_Avg_Salary  Above_Avg
0  Sumed     Data   60000     71666.666667      False
1  Priya       HR   45000     47500.000000      False
2  Rahul     Data   75000     71666.666667       True
3   Neha  Finance   55000     55000.000000      False
4  Arjun     Data   80000     71666.666667       True
5  Divya       HR   50000     47500.000000       True


*transform() returned 6 rows (same as original) — not 3 rows like agg() would. Each row got its group's average broadcast back to it. This is one of the most powerful GroupBy patterns in real analysis.*

## FILTER(): keep only Groups meeting a condition

In [22]:
# filter() on a GroupBy — keeps entire GROUPS that meet a condition
# NOT the same as df[condition] which filters individual rows
# This filters at GROUP level

# Keep only departments where mean salary > 50000
df.groupby('Dept').filter(lambda x: x['Salary'].mean() > 50000)



,Name,Dept,Salary,Exp,Level,Dept_Avg_Salary,Above_Avg
0,Sumed,Data,60000,3,Junior,71666.666667,False
2,Rahul,Data,75000,5,Senior,71666.666667,True
3,Neha,Finance,55000,4,Junior,55000.000000,False
4,Arjun,Data,80000,6,Senior,71666.666667,True


In [23]:
# HOW it works:
# lambda x → x is the entire sub-DataFrame for each group
# x['Salary'].mean() calculates mean for that group
# if True → ALL rows of that group are kept
# if False → ALL rows of that group are dropped

# Keep only depts with more than 1 employee
df.groupby('Dept').filter(lambda x: len(x) > 1)

,Name,Dept,Salary,Exp,Level,Dept_Avg_Salary,Above_Avg
0,Sumed,Data,60000,3,Junior,71666.666667,False
1,Priya,HR,45000,2,Junior,47500.000000,False
2,Rahul,Data,75000,5,Senior,71666.666667,True
4,Arjun,Data,80000,6,Senior,71666.666667,True
5,Divya,HR,50000,3,Junior,47500.000000,True


## RESET INDEX(): reset index after groupby

In [24]:
# After groupby + agg, the group key becomes the index
# This makes further operations awkward
# reset_index() converts the index back to a regular column

result = df.groupby('Dept')['Salary'].mean()
print(result.index)   # Index(['Data','Finance','HR']) ← group key is index

# Fix: reset_index() → Dept becomes a regular column again
result = result.reset_index()
print(result)         # now a clean DataFrame with Dept and Salary columns

# One-liner — chain reset_index() directly
result = df.groupby('Dept')['Salary'].mean().reset_index()

Index(['Data', 'Finance', 'HR'], dtype='object', name='Dept')
      Dept        Salary
0     Data  71666.666667
1  Finance  55000.000000
2       HR  47500.000000


*Always chain .reset_index() after groupby+agg in real code — unless you specifically need the group key as index for further operations. It keeps your output clean and usable.*

## Everything You Must Know Cold

**Split → Apply → Combine** — the mental model behind every GroupBy operation. Internalize this pattern.

**groupby()** returns a GroupBy object — not a DataFrame. No computation happens yet. This is lazy evaluation. Computation only triggers when you call *.mean(), .sum(), .agg()* etc.

**agg()**— the professional way to apply multiple functions at once. Named aggregation syntax *new_col = ('source_col', 'func')* gives clean output column names.

**transform()** — unlike *agg()* which shrinks rows, *transform()* keeps the same number of rows by broadcasting group results back. Essential for adding group statistics as a new column.

**filter()** — filters at group level, not row level. Entire groups are kept or dropped based on a condition applied to the whole group.

**as_index=False** or **.reset_index()** — after groupby, the group key becomes the index. Always reset it to keep your output as a clean usable DataFrame.

**numeric_only=True** — use in newer Pandas versions to avoid FutureWarnings when calling *.mean()* or *.sum()* on a GroupBy without specifying a column.

 ## When to use
 
 **agg vs transform vs filter** — The Most Important Distinction

**agg()** Shrinks — one row per groupYou want summary statistics

**transform()** Same as original — one row per original rowYou want to add group stats back to original df

**filter()** Subset of original rowsYou want to keep/drop entire groups